#Basic Tasks

###1. Install and authenticate the Databricks CLI
- databricks -v
- databricks auth login --host <workspace_url> --profile workspace1
- databricks auth profiles

###2. Initialize a Declarative Automation Bundle project

- databricks bundle init --profile my_workspace1

Creates databricks.yml, resources/sample_job.job.yml (one job: sample_job, notebook task), and src/.

###3. Run databricks bundle validate and databricks bundle deploy -t dev

- cd dabs1
- databricks bundle validate --profile my_workspace1
- databricks bundle deploy --profile my_workspace1 -t dev
- databricks bundle run sample_job --profile my_workspace1 -t dev

"[dev rinkimehrasalesforce] sample_job" TERMINATED SUCCESS

#Intermediate Tasks

###4
Add another target in databricks.yml with a different workspace host and configure the Job to run as a service principal.

targets:
  dev:
    mode: development
    default: true
    workspace:
      host: https://<dev-workspace-url>

  prod:
    workspace:
      host: https://<prod-workspace-url>
    run_as:
      service_principal_name: <service_principal_id>

###5
- Created a service principal in the workspace under Identity and access
- Generated a client secret for it
- Added a prod target in databricks.yml pointing to a different host, with run_as set to the service principal
- Set up a CLI profile authenticated via M2M using the service principal instead of personal login
- Ran databricks bundle deploy with that profile to prod, deployment completed successfully
- Confirmed the job was created under the service principal using databricks jobs list

###6
name: Validate Bundle

on:
  pull_request:
    branches: [main]

jobs:
  validate:
    runs-on: ubuntu-latest
    steps:
      - name: Checkout code
        uses: actions/checkout@v4

      - name: Install Databricks CLI
        uses: databricks/setup-cli@main

      - name: Validate bundle
        run: databricks bundle validate
        env:
          DATABRICKS_HOST: ${{ secrets.DATABRICKS_HOST }}
          DATABRICKS_CLIENT_ID: ${{ secrets.DATABRICKS_CLIENT_ID }}
          DATABRICKS_CLIENT_SECRET: ${{ secrets.DATABRICKS_CLIENT_SECRET }}

#Advanced Tasks

###7
name: Validate and Deploy

on:
  pull_request:
    branches: [main]
  push:
    branches: [main]

permissions:
  id-token: write
  contents: read

jobs:
  validate:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: databricks/setup-cli@main
      - name: Validate bundle
        run: databricks bundle validate
        env:
          DATABRICKS_HOST: ${{ secrets.DATABRICKS_HOST }}
          DATABRICKS_CLIENT_ID: ${{ secrets.DATABRICKS_CLIENT_ID }}
          DATABRICKS_CLIENT_SECRET: ${{ secrets.DATABRICKS_CLIENT_SECRET }}

  deploy:
    if: github.ref == 'refs/heads/main' && github.event_name == 'push'
    needs: validate
    runs-on: ubuntu-latest
    permissions:
      id-token: write
      contents: read
    steps:
      - uses: actions/checkout@v4
      - uses: databricks/setup-cli@main
      - name: Deploy to prod via OIDC
        run: databricks bundle deploy -t prod
        env:
          DATABRICKS_HOST: ${{ secrets.DATABRICKS_HOST }}
          ARM_USE_OIDC: true

###8
1. Check the job — run it, confirm it's actually broken
-databricks bundle run <job_name> -t prod

2. Find the last working commit
-git log --oneline

3. Get the old working files back
-git checkout <good-commit-hash> -- databricks.yml resources/

4. Redeploy the old version
-databricks bundle deploy -t prod

5. Confirm it works again
-databricks bundle run <job_name> -t prod

6. Revert the bad commit in git (so history stays clean)
-git revert <bad-commit-hash>
-git push origin main

###9
1. Local Setup (one-time)
Install Databricks CLI on your machine
Authenticate: databricks auth login --host <dev-url> --profile <your-name> (U2M — browser login)
Confirm: databricks current-user me
2. Edit the Bundle

Our pipeline is code, structured as a Databricks Asset Bundle:

databricks.yml — main config (bundle name, targets, variables)
resources/*.yml — job/pipeline definitions
src/ — notebooks and scripts

Edit these locally using VS Code (not Notepad — YAML is indentation-sensitive).

3. Test Locally Before Pushing
databricks bundle validate --profile <your-name>
databricks bundle deploy --profile <your-name> -t dev
databricks bundle run <job_name> --profile <your-name> -t dev

Dev deploys are prefixed with your username ([dev yourname] job_name), so testing never collides with teammates.

4. Open a Pull Request

Pushing your branch and opening a PR against main triggers CI:

databricks bundle validate

This only checks the bundle is valid — nothing gets deployed at this stage.

5. Merge to Main → Automatic Prod Deploy

On merge, CI/CD runs:

databricks bundle deploy -t prod

using OIDC authentication — no stored secrets. GitHub proves its identity with a short-lived token, and the job runs under a service principal, not a personal account, keeping prod stable and independent of any one person.

6. If Something Breaks
databricks bundle run <job_name> -t prod          # confirm it's broken
git checkout <last-good-commit> -- databricks.yml resources/
databricks bundle deploy -t prod                   # rollback
git revert <bad-commit>
git push origin main   